In [2]:
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Set, Optional
import logging
from collections import defaultdict
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


class ProgressiveAutocompleteEvaluator:
    """
    Évaluateur pour système d'autocomplétion PROGRESSIVE
    Simule la frappe caractère par caractère
    """
    
    def __init__(self):
        logger.info(f"✅ Évaluateur d'autocomplétion progressive initialisé")
    
    def load_and_analyze_dataset(self, excel_path: str) -> Dict[str, Set[str]]:
        """Charge le dataset verbe -> compléments"""
        if not os.path.exists(excel_path):
            raise FileNotFoundError(f"❌ Fichier introuvable : {excel_path}")
        
        try:
            df = pd.read_excel(excel_path, engine='openpyxl')
        except:
            try:
                df = pd.read_excel(excel_path, engine='xlrd')
            except Exception as e:
                raise Exception(f"❌ Erreur lecture Excel: {e}")
        
        logger.info(f"📂 Fichier chargé: {len(df)} lignes")
        
        # Détecter colonnes
        verbe_col = None
        complement_col = None
        
        for col in df.columns:
            col_lower = str(col).lower()
            if 'verb' in col_lower:
                verbe_col = col
            elif 'complement' in col_lower or 'complément' in col_lower:
                complement_col = col
        
        if verbe_col is None or complement_col is None:
            verbe_col = df.columns[0]
            complement_col = df.columns[1]
        
        logger.info(f"🔍 Colonnes: Verbe='{verbe_col}', Complément='{complement_col}'")
        
        # Construction dictionnaire
        verb_to_complements = defaultdict(set)
        
        for idx, row in df.iterrows():
            try:
                verbe = str(row[verbe_col]).strip().lower()
                complement = str(row[complement_col]).strip()
                
                if verbe and complement and verbe != 'nan' and complement != 'nan':
                    verb_to_complements[verbe].add(complement)
            except:
                continue
        
        # Stats
        print("\n" + "=" * 80)
        print("📊 ANALYSE DU DATASET")
        print("=" * 80)
        print(f"Total verbes: {len(verb_to_complements)}")
        print(f"Total associations: {sum(len(v) for v in verb_to_complements.values())}")
        
        top_verbs = sorted(verb_to_complements.items(), 
                          key=lambda x: len(x[1]), 
                          reverse=True)[:10]
        
        print("\n🎯 TOP 10 VERBES:")
        for verbe, complements in top_verbs:
            print(f"  '{verbe}': {len(complements)} compléments")
            for comp in list(complements)[:2]:
                print(f"    - {comp}")
        
        return dict(verb_to_complements)
    
    def evaluate_progressive_autocomplete(
        self,
        verb_to_complements: Dict[str, Set[str]],
        autocomplete_function,
        test_verbs: Optional[List[str]] = None,
        max_verbs: int = None,
        typing_steps: List[str] = None
    ) -> Dict[str, Any]:
        """
        Évalue un système d'autocomplétion PROGRESSIVE
        
        Args:
            verb_to_complements: Dict verbe -> compléments
            autocomplete_function: Fonction(query: str) -> List[str] de suggestions
            test_verbs: Verbes à tester
            max_verbs: Limite de verbes
            typing_steps: Points de frappe à tester (ex: après verbe, après verbe + début complément)
        """
        if test_verbs is None:
            test_verbs = list(verb_to_complements.keys())
        
        if max_verbs:
            test_verbs = test_verbs[:max_verbs]
        
        logger.info(f"🚀 Évaluation progressive sur {len(test_verbs)} verbes...")
        
        results = {
            'by_verb': {},
            'by_typing_stage': defaultdict(list),
            'global_metrics': defaultdict(list),
            'detailed_examples': [],
            'errors': []
        }
        
        for i, verbe in enumerate(test_verbs):
            if verbe not in verb_to_complements:
                continue
            
            if (i + 1) % 10 == 0:
                logger.info(f"📊 Traité {i + 1}/{len(test_verbs)} verbes...")
            
            expected_complements = list(verb_to_complements[verbe])
            
            try:
                # TEST 1: Après le verbe seul
                query_stage1 = verbe
                suggestions_stage1 = self._normalize_suggestions(
                    autocomplete_function(query_stage1)
                )
                
                metrics_stage1 = self._calculate_metrics(
                    expected_complements,
                    suggestions_stage1,
                    top_k_values=[1, 3, 5, 10]
                )
                metrics_stage1['stage'] = 'after_verb'
                metrics_stage1['query'] = query_stage1
                
                # TEST 2: Après verbe + début du premier complément
                metrics_stage2 = None
                suggestions_stage2 = []
                if expected_complements:
                    first_complement = expected_complements[0].lower()
                    # Extraire les premiers mots après le verbe
                    words_after_verb = first_complement.replace(verbe, '').strip().split()
                    
                    if words_after_verb:
                        # Prendre les 2 premiers mots
                        partial_complement = ' '.join(words_after_verb[:2])
                        query_stage2 = f"{verbe} {partial_complement}"
                        
                        suggestions_stage2 = self._normalize_suggestions(
                            autocomplete_function(query_stage2)
                        )
                        
                        metrics_stage2 = self._calculate_metrics(
                            expected_complements,
                            suggestions_stage2,
                            top_k_values=[1, 3, 5, 10]
                        )
                        metrics_stage2['stage'] = 'after_partial'
                        metrics_stage2['query'] = query_stage2
                
                # TEST 3: Simulation de frappe progressive (caractère par caractère)
                progressive_metrics = []
                if expected_complements:
                    sample_complement = expected_complements[0]
                    # Tester à différentes longueurs de frappe
                    for length in [len(verbe) + 3, len(verbe) + 6, len(verbe) + 10]:
                        if length < len(sample_complement):
                            partial_query = sample_complement[:length].lower()
                            suggestions = self._normalize_suggestions(
                                autocomplete_function(partial_query)
                            )
                            
                            metrics_progressive = self._calculate_metrics(
                                expected_complements,
                                suggestions,
                                top_k_values=[1, 3, 5]
                            )
                            metrics_progressive['stage'] = f'progressive_{length}chars'
                            metrics_progressive['query'] = partial_query
                            progressive_metrics.append(metrics_progressive)
                
                # Stocker les résultats
                verb_result = {
                    'verbe': verbe,
                    'expected_count': len(expected_complements),
                    'stage1': metrics_stage1,
                    'stage2': metrics_stage2,
                    'progressive': progressive_metrics
                }
                
                results['by_verb'][verbe] = verb_result
                
                # Agréger par stade (à faire après la boucle, pas ici)
                results['by_typing_stage']['after_verb'].append(metrics_stage1)
                if metrics_stage2:
                    results['by_typing_stage']['after_partial'].append(metrics_stage2)
                
                for prog_met in progressive_metrics:
                    results['by_typing_stage']['progressive'].append(prog_met)
                
                # Exemple détaillé
                if i < 5:  # Garder 5 premiers exemples
                    stage2_suggestions = suggestions_stage2 if metrics_stage2 else []
                    results['detailed_examples'].append({
                        'verbe': verbe,
                        'expected': expected_complements[:3],
                        'stage1_query': query_stage1,
                        'stage1_suggestions': suggestions_stage1[:5],
                        'stage2_query': metrics_stage2['query'] if metrics_stage2 else None,
                        'stage2_suggestions': stage2_suggestions[:5],
                        'metrics': {
                            'stage1_recall@5': metrics_stage1.get('recall@5', 0),
                            'stage2_recall@5': metrics_stage2.get('recall@5', 0) if metrics_stage2 else 0
                        }
                    })
                
            except Exception as e:
                logger.error(f"❌ Erreur avec '{verbe}': {e}")
                results['errors'].append({'verbe': verbe, 'error': str(e)})
        
        # Calculer moyennes par stade (après la boucle)
        self._calculate_stage_averages(results)
        
        logger.info(f"✅ Évaluation terminée: {len(results['by_verb'])} verbes")
        
        return results
    
    def _normalize_suggestions(self, suggestions: Any) -> List[str]:
        """Normalise les suggestions"""
        if not suggestions:
            return []
        
        normalized = []
        
        if isinstance(suggestions, list):
            for sugg in suggestions:
                if isinstance(sugg, dict):
                    text = (sugg.get('texte', '') or 
                           sugg.get('contenu', '') or 
                           sugg.get('complement', '') or
                           sugg.get('content', ''))
                    if text:
                        normalized.append(str(text).strip().lower())
                elif isinstance(sugg, str):
                    normalized.append(sugg.strip().lower())
        elif isinstance(suggestions, str):
            normalized.append(suggestions.strip().lower())
        
        # Dédupliquer
        seen = set()
        result = []
        for item in normalized:
            if item and item not in seen:
                seen.add(item)
                result.append(item)
        
        return result
    
    def _calculate_metrics(
        self,
        expected: List[str],
        suggestions: List[str],
        top_k_values: List[int]
    ) -> Dict[str, float]:
        """Calcule les métriques"""
        metrics = {
            'suggestions_count': len(suggestions),
            'expected_count': len(expected)
        }
        
        expected_lower = [e.lower().strip() for e in expected]
        
        # Pour chaque k
        for k in top_k_values:
            top_k = suggestions[:k]
            
            # Recall@k: combien d'attendus sont dans top-k ?
            found = sum(1 for exp in expected_lower if self._is_match(exp, top_k))
            recall = found / len(expected) if expected else 0
            metrics[f'recall@{k}'] = recall
            
            # Precision@k: combien de top-k sont valides ?
            valid = sum(1 for sugg in top_k if self._is_match(sugg, expected_lower))
            precision = valid / len(top_k) if top_k else 0
            metrics[f'precision@{k}'] = precision
            
            # F1@k
            if recall + precision > 0:
                metrics[f'f1@{k}'] = 2 * (recall * precision) / (recall + precision)
            else:
                metrics[f'f1@{k}'] = 0
        
        # MRR: position du premier match
        for rank, sugg in enumerate(suggestions, 1):
            if self._is_match(sugg, expected_lower):
                metrics['mrr'] = 1.0 / rank
                metrics['first_match_rank'] = rank
                break
        else:
            metrics['mrr'] = 0
            metrics['first_match_rank'] = -1
        
        return metrics
    
    def _is_match(self, text: str, candidates: List[str], threshold: float = 0.80) -> bool:
        """Vérifie si un texte correspond à un candidat"""
        text_clean = text.lower().strip()
        
        for candidate in candidates:
            candidate_clean = candidate.lower().strip()
            
            # Match exact
            if text_clean == candidate_clean:
                return True
            
            # L'un contient l'autre (pour autocomplétion progressive)
            if text_clean in candidate_clean or candidate_clean in text_clean:
                return True
            
            # Similarité de séquence
            if self._string_similarity(text_clean, candidate_clean) >= threshold:
                return True
        
        return False
    
    def _string_similarity(self, s1: str, s2: str) -> float:
        """Calcule similarité entre 2 strings"""
        from difflib import SequenceMatcher
        return SequenceMatcher(None, s1, s2).ratio()
    
    def _calculate_stage_averages(self, results: Dict[str, Any]):
        """Calcule les moyennes par stade de frappe"""
        # Faire une copie des clés pour éviter de modifier pendant l'itération
        stage_keys = list(results['by_typing_stage'].keys())
        
        for stage_name in stage_keys:
            metrics_list = results['by_typing_stage'][stage_name]
            if not metrics_list:
                continue
            
            # Ne pas calculer pour les moyennes existantes
            if stage_name.endswith('_averages'):
                continue
            
            # Calculer moyennes pour chaque métrique
            avg_metrics = defaultdict(list)
            
            for metrics in metrics_list:
                for key, value in metrics.items():
                    if isinstance(value, (int, float)) and key not in ['stage', 'query', 'suggestions_count', 'expected_count']:
                        avg_metrics[key].append(value)
            
            # Ajouter les moyennes dans un nouveau champ
            results['by_typing_stage'][f'{stage_name}_averages'] = {
                f'avg_{key}': np.mean(values) 
                for key, values in avg_metrics.items()
            }
    
    def generate_report(self, results: Dict[str, Any], output_path: str = None) -> str:
        """Génère un rapport pour autocomplétion progressive"""
        lines = []
        lines.append("=" * 80)
        lines.append("📊 RAPPORT D'ÉVALUATION - AUTOCOMPLÉTION PROGRESSIVE")
        lines.append("=" * 80)
        lines.append("\n💡 Méthodologie:")
        lines.append("  Ce système simule la frappe progressive de l'utilisateur")
        lines.append("  et mesure la qualité des suggestions à différents stades.")
        
        # Métriques par stade
        lines.append("\n" + "=" * 80)
        lines.append("🎯 PERFORMANCE PAR STADE DE FRAPPE")
        lines.append("=" * 80)
        
        stages_info = [
            ('after_verb', 'Après le verbe seul (ex: "remplacer")'),
            ('after_partial', 'Après verbe + début complément (ex: "remplacer le dis")'),
            ('progressive', 'Frappe progressive (simulation caractère par caractère)')
        ]
        
        for stage_key, stage_desc in stages_info:
            avg_key = f'{stage_key}_averages'
            if avg_key in results['by_typing_stage']:
                avg_metrics = results['by_typing_stage'][avg_key]
                
                lines.append(f"\n📍 {stage_desc}")
                lines.append("-" * 60)
                
                for k in [1, 3, 5, 10]:
                    recall_key = f'avg_recall@{k}'
                    precision_key = f'avg_precision@{k}'
                    f1_key = f'avg_f1@{k}'
                    
                    if recall_key in avg_metrics:
                        lines.append(f"  Recall@{k:2d}:     {avg_metrics[recall_key]:6.1%}")
                    if precision_key in avg_metrics:
                        lines.append(f"  Precision@{k:2d}:  {avg_metrics[precision_key]:6.1%}")
                    if f1_key in avg_metrics:
                        lines.append(f"  F1-Score@{k:2d}:  {avg_metrics[f1_key]:6.1%}")
                
                if 'avg_mrr' in avg_metrics:
                    lines.append(f"  MRR:            {avg_metrics['avg_mrr']:.3f}")
        
        # Exemples détaillés
        lines.append("\n" + "=" * 80)
        lines.append("🔍 EXEMPLES DÉTAILLÉS")
        lines.append("=" * 80)
        
        for ex in results['detailed_examples'][:5]:
            lines.append(f"\n📝 Verbe: '{ex['verbe']}'")
            lines.append(f"   Compléments attendus (3 premiers):")
            for comp in ex['expected']:
                lines.append(f"     ✓ {comp}")
            
            lines.append(f"\n   STADE 1: Requête = '{ex['stage1_query']}'")
            lines.append(f"   Suggestions (top 5):")
            for i, sugg in enumerate(ex['stage1_suggestions'], 1):
                # Vérifier si c'est un match
                is_match = self._is_match(sugg, [e.lower() for e in ex['expected']])
                marker = "✓" if is_match else "✗"
                lines.append(f"     {marker} {i}. {sugg}")
            lines.append(f"   Recall@5: {ex['metrics']['stage1_recall@5']:.1%}")
            
            if ex['stage2_query']:
                lines.append(f"\n   STADE 2: Requête = '{ex['stage2_query']}'")
                lines.append(f"   Suggestions (top 5):")
                for i, sugg in enumerate(ex['stage2_suggestions'], 1):
                    is_match = self._is_match(sugg, [e.lower() for e in ex['expected']])
                    marker = "✓" if is_match else "✗"
                    lines.append(f"     {marker} {i}. {sugg}")
                lines.append(f"   Recall@5: {ex['metrics']['stage2_recall@5']:.1%}")
        
        # Analyse comparative
        lines.append("\n" + "=" * 80)
        lines.append("📈 ANALYSE COMPARATIVE")
        lines.append("=" * 80)
        
        # Comparer stage1 vs stage2
        if 'after_verb_averages' in results['by_typing_stage'] and 'after_partial_averages' in results['by_typing_stage']:
            avg1 = results['by_typing_stage']['after_verb_averages']
            avg2 = results['by_typing_stage']['after_partial_averages']
            
            recall1 = avg1.get('avg_recall@5', 0)
            recall2 = avg2.get('avg_recall@5', 0)
            
            lines.append(f"\nComparaison Recall@5:")
            lines.append(f"  Après verbe seul:          {recall1:.1%}")
            lines.append(f"  Après verbe + début:       {recall2:.1%}")
            
            if recall2 > recall1:
                improvement = ((recall2 - recall1) / recall1 * 100) if recall1 > 0 else 0
                lines.append(f"  Amélioration:              +{improvement:.1f}%")
                lines.append(f"\n✅ Le système s'améliore avec plus de contexte (bon signe !)")
            else:
                lines.append(f"\n⚠️  Le système ne s'améliore pas avec plus de contexte")
        
        # Recommandations
        lines.append("\n" + "=" * 80)
        lines.append("💡 RECOMMANDATIONS")
        lines.append("=" * 80)
        
        if 'after_verb_averages' in results['by_typing_stage']:
            avg_recall = results['by_typing_stage']['after_verb_averages'].get('avg_recall@5', 0)
            
            if avg_recall >= 0.5:
                lines.append("\n✅ EXCELLENTES PERFORMANCES")
                lines.append("   • Le système fonctionne bien dès le verbe")
                lines.append("   • Continuer à enrichir la base de données")
                lines.append("   • Surveiller les verbes avec faible score")
            elif avg_recall >= 0.3:
                lines.append("\n⚡ BONNES PERFORMANCES")
                lines.append("   • Le système est efficace avec le contexte")
                lines.append("   • Améliorer le filtrage initial (après verbe seul)")
                lines.append("   • Tester différents seuils de similarité")
            else:
                lines.append("\n⚠️  PERFORMANCES À AMÉLIORER")
                lines.append("   • Vérifier que le VectorStore contient bien toutes les données")
                lines.append("   • Augmenter k_documents dans la recherche")
                lines.append("   • Réduire min_similarity pour plus de résultats")
                lines.append("   • Vérifier la qualité des embeddings")
        
        lines.append("\n🔧 OPTIMISATIONS SPÉCIFIQUES:")
        lines.append("   • Implémenter un cache pour verbes fréquents")
        lines.append("   • Pré-filtrer par préfixe avant recherche vectorielle")
        lines.append("   • Combiner filtrage texte + similarité sémantique")
        lines.append("   • Utiliser le contexte (localisation) pour affiner")
        
        if results.get('errors'):
            lines.append(f"\n❌ ERREURS: {len(results['errors'])}")
            for err in results['errors'][:5]:
                lines.append(f"  '{err['verbe']}': {err['error']}")
        
        lines.append("\n" + "=" * 80)
        
        report = "\n".join(lines)
        
        if output_path:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(report)
            logger.info(f"📄 Rapport sauvegardé: {output_path}")
        
        return report

    def create_quick_test_script(self):
        """Crée un script rapide pour tester votre système"""
        script = '''
#!/usr/bin/env python3
"""
Script de test rapide pour votre système d'autocomplétion
"""

import sys

def test_your_system():
    """Teste votre vrai système d'autocomplétion"""
    
    # ⚠️ IMPORTEZ VOTRE VRAI SYSTÈME ICI
    try:
        from suggestion_engine import get_suggestion_engine
        from langchain_groq import ChatGroq
        from vector_store import VectorStore
        
        print("🔧 Initialisation de votre système...")
        
        # Initialiser le LLM
        llm = ChatGroq(
            temperature=0.1,
            model_name="mixtral-8x7b-32768",
            groq_api_key="VOTRE_CLÉ_API"  # ⚠️ Remplacez
        )
        
        # Initialiser le VectorStore
        vectorstore = VectorStore()
        
        # Créer le moteur
        engine = get_suggestion_engine(vectorstore=vectorstore, llm=llm)
        
        # Fonction de test
        def autocomplete(query: str):
            suggestions = engine.get_suggestions(query)
            # Normaliser les suggestions
            normalized = []
            for sugg in suggestions:
                if isinstance(sugg, dict):
                    text = sugg.get('texte', sugg.get('contenu', sugg.get('complement', '')))
                    if text:
                        normalized.append(text.strip())
                else:
                    normalized.append(str(sugg).strip())
            return normalized
        
        return autocomplete
        
    except ImportError as e:
        print(f"❌ Erreur d'import: {e}")
        print("⚠️ Mode simulation activé")
        return None

def main():
    """Test interactif"""
    print("=" * 60)
    print("🧪 TEST INTERACTIF - AUTOCOMPLÉTION PROGRESSIVE")
    print("=" * 60)
    
    # Essayer de charger votre système
    autocomplete_func = test_your_system()
    
    if autocomplete_func is None:
        print("\\n📝 Mode simulation - entrez des suggestions manuellement")
        print("   (Pour utiliser votre vrai système, installez les modules)")
        
        def simulation_autocomplete(query: str):
            print(f"\\n🔍 Requête: '{query}'")
            print("💡 Suggestions (entrez une par ligne, vide pour terminer):")
            
            suggestions = []
            while True:
                sugg = input("  > ").strip()
                if not sugg:
                    break
                suggestions.append(sugg)
            
            return suggestions
        
        autocomplete_func = simulation_autocomplete
    
    # Tests pré-définis
    test_queries = [
        "remplacer",
        "remplacer le",
        "remplacer le disjoncteur",
        "vérifier",
        "vérifier la",
        "vérifier la terre",
        "installer",
        "installer une",
        "installer une protection"
    ]
    
    print("\\n🎯 TESTS PRÉ-DÉFINIS:")
    for query in test_queries:
        print(f"\\n📝 Requête: '{query}'")
        suggestions = autocomplete_func(query)[:5]
        
        if suggestions:
            print("   Suggestions (top 5):")
            for i, sugg in enumerate(suggestions, 1):
                print(f"     {i}. {sugg[:60]}...")
        else:
            print("   Aucune suggestion")
    
    # Mode interactif
    print("\\n" + "=" * 60)
    print("⌨️  MODE INTERACTIF (tapez 'quit' pour quitter)")
    print("=" * 60)
    
    while True:
        query = input("\\nEntrez votre requête: ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            break
        
        if not query:
            continue
        
        suggestions = autocomplete_func(query)
        
        print(f"\\n🔍 Requête: '{query}'")
        print(f"📊 {len(suggestions)} suggestions trouvées:")
        
        for i, sugg in enumerate(suggestions[:10], 1):
            print(f"  {i}. {sugg}")
        
        if len(suggestions) > 10:
            print(f"  ... et {len(suggestions) - 10} autres")

if __name__ == "__main__":
    main()
'''
        
        with open("test_autocomplete.py", "w", encoding="utf-8") as f:
            f.write(script)
        
        print("\n✅ Script de test créé: test_autocomplete.py")
        print("📝 Utilisation: python test_autocomplete.py")


def main():
    """Point d'entrée"""
    print("=" * 80)
    print("🤖 ÉVALUATION AUTOCOMPLÉTION PROGRESSIVE")
    print("=" * 80)
    
    # Configuration
    EXCEL_PATH = "/home/student24/Downloads/Mes projets/PROJET RAPPORTS/Rapport/LEL24 V1 (02-2013).xls"
    
    if not os.path.exists(EXCEL_PATH):
        print(f"❌ Fichier introuvable: {EXCEL_PATH}")
        # Créer un fichier de test
        print("Création d'un fichier de test de démonstration...")
        return
    
    # Initialiser
    evaluator = ProgressiveAutocompleteEvaluator()
    
    # Charger données
    print("\n📂 Chargement du dataset...")
    try:
        verb_data = evaluator.load_and_analyze_dataset(EXCEL_PATH)
    except Exception as e:
        print(f"❌ Erreur: {e}")
        return
    
    # Fonction RAG de simulation
    def simulation_autocomplete(query: str) -> List[str]:
        """Simule votre système d'autocomplétion"""
        query_lower = query.lower().strip()
        
        # Extraire le verbe (premier mot)
        words = query_lower.split()
        if not words:
            return []
        
        verbe = words[0]
        reste = ' '.join(words[1:]) if len(words) > 1 else ''
        
        # Récupérer compléments du verbe
        if verbe not in verb_data:
            return []
        
        complements = list(verb_data[verbe])
        
        # Filtrer par le reste de la requête
        if reste:
            filtered = []
            for comp in complements:
                comp_lower = comp.lower()
                # Vérifier si le complément commence par ou contient le reste
                if reste in comp_lower or comp_lower.startswith(reste):
                    filtered.append(comp)
            return filtered[:20]
        
        # Sinon retourner tous les compléments
        return complements[:20]
    
    # POUR UTILISER VOTRE VRAI SYSTÈME, décommentez ce code:
    """
    try:
        from suggestion_engine import get_suggestion_engine
        from langchain_groq import ChatGroq
        from vector_store import VectorStore
        
        print("🔧 Initialisation de votre système RAG...")
        
        llm = ChatGroq(temperature=0.1, model_name="mixtral-8x7b-32768", groq_api_key="VOTRE_CLÉ")
        vectorstore = VectorStore()
        engine = get_suggestion_engine(vectorstore=vectorstore, llm=llm)
        
        def real_autocomplete(query: str) -> List[str]:
            try:
                return engine.get_suggestions(query)
            except Exception as e:
                logger.error(f"Erreur: {e}")
                return []
        
        autocomplete_function = real_autocomplete
        print("✅ Votre vrai système est utilisé")
        
    except ImportError as e:
        print(f"⚠️ Impossible d'importer vos modules: {e}")
        print("📝 Utilisation de la simulation")
        autocomplete_function = simulation_autocomplete
    """
    
    # Utiliser la simulation pour l'instant
    autocomplete_function = simulation_autocomplete
    print("📝 Mode simulation activé")
    
    # Sélectionner verbes à tester
    all_verbs = list(verb_data.keys())
    verb_freq = [(v, len(verb_data[v])) for v in all_verbs]
    verb_freq.sort(key=lambda x: x[1], reverse=True)
    test_verbs = [v for v, _ in verb_freq[:5]]  # Réduire à 5 pour aller plus vite
    
    print(f"\n🔍 VERBES À TESTER ({len(test_verbs)}):")
    for verbe in test_verbs:
        print(f"  '{verbe}': {len(verb_data[verbe])} compléments")
    
    # Lancer évaluation
    print(f"\n🚀 LANCEMENT DE L'ÉVALUATION...")
    print(f"Cette évaluation simule la frappe progressive pour {len(test_verbs)} verbes")
    print("=" * 60)
    
    results = evaluator.evaluate_progressive_autocomplete(
        verb_to_complements=verb_data,
        autocomplete_function=autocomplete_function,
        test_verbs=test_verbs
    )
    
    # Générer rapport
    report = evaluator.generate_report(results, output_path="rapport_progressive.txt")
    print("\n" + report)
    
    # Créer un script de test interactif
    evaluator.create_quick_test_script()
    
    print("\n" + "=" * 80)
    print("✅ ÉVALUATION TERMINÉE")
    print("=" * 80)
    print("\n🎯 PROCHAINES ÉTAPES:")
    print("1. Modifiez test_autocomplete.py avec votre vrai système")
    print("2. Testez avec: python test_autocomplete.py")
    print("3. Améliorez votre VectorStore si nécessaire")
    print("4. Re-exécutez cette évaluation")


if __name__ == "__main__":
    main()

2025-12-08 11:56:52,351 - INFO - ✅ Évaluateur d'autocomplétion progressive initialisé
2025-12-08 11:56:52,368 - INFO - 📂 Fichier chargé: 750 lignes
2025-12-08 11:56:52,369 - INFO - 🔍 Colonnes: Verbe='Verbe', Complément='Complément'
2025-12-08 11:56:52,433 - INFO - 🚀 Évaluation progressive sur 5 verbes...


🤖 ÉVALUATION AUTOCOMPLÉTION PROGRESSIVE

📂 Chargement du dataset...

📊 ANALYSE DU DATASET
Total verbes: 58
Total associations: 543

🎯 TOP 10 VERBES:
  'remplacer': 115 compléments
    - l'interrupteur différentiel par un disjoncteur différentiel assurant le pouvoir de coupure de
    - les tubes "IRO" par des conduits MRL relier au conducteur de protection
  'installer': 64 compléments
    - un relais homopolaire sur le départ de la cellule
    - un écran métallique plein en façade de la cellule
  'protéger': 46 compléments
    - les moteurs triphasés contre la perte d’une phase
    - contre les surintensités, par dispositifs à maximum de courant ou sondes thermiques, les machines tournantes du local ATEX
  'réaliser': 36 compléments
    - une liaison à isolation double ou renforcée
    - l'éclairage d'ambiance par au moins deux blocs autonomes
  'alimenter': 17 compléments
    - un seul matériel en aval du transformateur de séparation
    - par l'intermédiaire d'un transformateur de sé

2025-12-08 11:56:54,120 - INFO - ✅ Évaluation terminée: 5 verbes
2025-12-08 11:56:54,139 - INFO - 📄 Rapport sauvegardé: rapport_progressive.txt



📊 RAPPORT D'ÉVALUATION - AUTOCOMPLÉTION PROGRESSIVE

💡 Méthodologie:
  Ce système simule la frappe progressive de l'utilisateur
  et mesure la qualité des suggestions à différents stades.

🎯 PERFORMANCE PAR STADE DE FRAPPE

📍 Après le verbe seul (ex: "remplacer")
------------------------------------------------------------
  Recall@ 1:       2.7%
  Precision@ 1:  100.0%
  F1-Score@ 1:    5.1%
  Recall@ 3:       8.0%
  Precision@ 3:  100.0%
  F1-Score@ 3:   14.3%
  Recall@ 5:      17.6%
  Precision@ 5:  100.0%
  F1-Score@ 5:   28.6%
  Recall@10:      31.2%
  Precision@10:  100.0%
  F1-Score@10:   45.4%
  MRR:            1.000

📍 Après verbe + début complément (ex: "remplacer le dis")
------------------------------------------------------------
  Recall@ 1:       2.7%
  Precision@ 1:  100.0%
  F1-Score@ 1:    5.1%
  Recall@ 3:       5.0%
  Precision@ 3:  100.0%
  F1-Score@ 3:    9.3%
  Recall@ 5:       5.0%
  Precision@ 5:  100.0%
  F1-Score@ 5:    9.3%
  Recall@10:       5.6%
  Precisi